# Held-out DAS-versus-network comparison registration

This advisor-facing notebook verifies the comparison contract before any candidate-time row is parsed for matching. It reads registration metadata and byte hashes only. The contract retains all 22 frozen DAS-v2 candidates and all 33 frozen network candidates, uses the already registered eight-second deterministic rule within each interval, and preserves the 32-unit network evaluation ledger. It does **not** reveal a match result or assign an earthquake/repeater label.

In [ ]:
from pathlib import Path
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'config' / 'heldout_das_network_comparison.json').is_file()
)
REGISTRATION = ROOT / 'outputs' / 'heldout_v2' / 'registration'

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

config_path = ROOT / 'config' / 'heldout_das_network_comparison.json'
status_path = REGISTRATION / 'comparison_registration_status.json'
config = json.loads(config_path.read_text())
status = json.loads(status_path.read_text())

assert status['status'] == 'PASS'
assert status['comparison_config_sha256'] == sha256(config_path)
assert status['release_anchor']['DAS_freeze_checkpoint_commit_sha'] == (
    '28f58c3fa75d24688117554e3a1661e55318e87d'
)
assert status['release_anchor']['repository_visibility'] == 'private'
for name, declaration in config['frozen_inputs'].items():
    assert sha256(ROOT / declaration['path']) == declaration['sha256'], name
assert status['network_raw_candidate_count'] == 33
assert status['network_evaluation_unit_count'] == 32
assert status['DAS_v2_candidate_count'] == 22
assert status['registered_interval_count'] == 12
assert status['time_only_matching_window_s'] == 8.0
assert not status['cross_interval_matching_enabled']
assert status['unmatched_DAS_rows_retained']
assert status['unmatched_network_rows_retained']
assert status['schema_headers_verified_exact']
for field in [
    'network_candidate_rows_opened',
    'DAS_candidate_rows_opened',
    'network_evaluation_rows_opened',
    'network_adjudication_rows_opened',
    'network_candidate_time_fields_read',
    'DAS_candidate_time_fields_read',
    'catalog_association_rows_opened',
    'family_label_rows_opened',
    'comparison_output_products_present_before_registration',
]:
    assert status[field] == 0
print('PASS: hashes, schemas, inherited match rule, full retention, and zero row/time access verify')

In [ ]:
summary = pd.DataFrame({
    'registered quantity': [
        'held-out intervals',
        'duration (h)',
        'frozen DAS-v2 candidate rows',
        'frozen raw network candidate rows',
        'frozen network event units',
        'maximum absolute match difference (s)',
        'candidate/evaluation rows opened at registration',
        'candidate-time fields read at registration',
    ],
    'value': [
        status['registered_interval_count'],
        status['registered_total_duration_h'],
        status['DAS_v2_candidate_count'],
        status['network_raw_candidate_count'],
        status['network_evaluation_unit_count'],
        status['time_only_matching_window_s'],
        status['network_candidate_rows_opened'] + status['DAS_candidate_rows_opened']
        + status['network_evaluation_rows_opened'] + status['network_adjudication_rows_opened'],
        status['network_candidate_time_fields_read'] + status['DAS_candidate_time_fields_read'],
    ],
})
display(summary)
print('Matching:', status['time_only_matching_algorithm'])

In [ ]:
labels = ['DAS v2\nraw candidates', 'network\nraw candidates', 'network\nevent units']
values = [
    status['DAS_v2_candidate_count'],
    status['network_raw_candidate_count'],
    status['network_evaluation_unit_count'],
]
fig, ax = plt.subplots(figsize=(7, 3.7))
bars = ax.bar(labels, values, color=['tab:orange', 'tab:blue', 'tab:green'])
ax.bar_label(bars)
ax.set_ylabel('Frozen rows or units')
ax.set_title('Registered populations before time-only matching')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
plt.show()

In [ ]:
schema_summary = pd.DataFrame([
    {
        'frozen_table': name,
        'field_count': len(fields),
        'header_verified': status['verified_schema_fields'][name] == fields,
    }
    for name, fields in config['frozen_schema_headers'].items()
])
display(schema_summary)
print('Catalog fields allowed in time matching:', config['time_only_matching']['catalog_fields_allowed'])
print('Family fields allowed in time matching:', config['time_only_matching']['family_fields_allowed'])

In [ ]:
# Display-only advisor controls. They select registration metadata, never candidate rows.
DISPLAY_SECTION = 'time_only_matching'  # try 'post_time_only_network_context' or 'detector_policy'
SHOW_FROZEN_INPUT_HASHES = False

display(pd.Series(config[DISPLAY_SECTION], name=DISPLAY_SECTION).to_frame())
if SHOW_FROZEN_INPUT_HASHES:
    display(pd.DataFrame([
        {'name': name, 'path': item['path'], 'sha256': item['sha256']}
        for name, item in config['frozen_inputs'].items()
    ]))
print('Current gate:', status['candidate_time_table_access_gate'])
print('Next gate:', status['next_stage_gate'])

## Decision and next gate

The comparison is specified but has not run. The eight-second window is inherited from the pre-held-out DAS contract and frozen network union; it was not selected after seeing held-out matches. Matching must occur independently within each interval, maximize pair count before minimizing total absolute time difference, and retain every matched and unmatched row.

Candidate-time access remains **STOP** until the runner and its failure-path tests are committed, pushed to the private remote, and released against an exact remote SHA. The runner must checksum the time-only output before opening network adjudication/evaluation rows. Even after matching, a DAS-only row remains pending independent catalog, forced-network-score, DAS-artifact, and waveform adjudication; it is not automatically an earthquake, repeater, or extension.